### Для перевода описания

In [32]:
pip install deep_translator

## Импорт библиотек

In [34]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd

In [35]:
url = "https://ru.wikipedia.org/wiki/250_лучших_фильмов_по_версии_IMDb"

In [36]:
page = requests.get(url)

In [37]:
page.status_code

200

In [38]:
soup = bs(page.text, 'html.parser')

## Парсинг данных

In [40]:
films_tr = soup.find_all("tr")[1:]
films_data = []

for row in films_tr:
    td_list = row.find_all("td")
    
    if len(td_list) < 5:
        continue

    films_name_raw = td_list[1].text.strip()
    films_name_wiki = films_name_raw.replace(' ', '_')
    url_for_description = f"https://ru.wikipedia.org/wiki/{films_name_wiki}"
    
    try:
        souper = requests.get(url_for_description, timeout=5)
        soup_description = bs(souper.text, "html.parser")
        paragraphs = soup_description.find_all("p")
        description_text = "Нет описания"
        for p in paragraphs:
            text = p.text.strip()
            if text and not text.lower().startswith("эта статья"):
                description_text = text
                break
    except requests.exceptions.RequestException:
        description_text = "None"

    films = {'Название': films_name_raw, 'Год': td_list[2].text.strip(),'Режиссёр': td_list[3].text.strip(), 'Рейтинг': td_list[4].text.strip(),'Описание': description_text}

    films_data.append(films)

df = pd.DataFrame(films_data)

In [41]:
df.head()

,Название,Год,Режиссёр,Рейтинг,Описание
0,Побег из Шоушенка,1994,Фрэнк Дарабонт,"эпический фильм, историческая драма, тюремная ...",«Побе́г из Шоуше́нка» (англ. The Shawshank Red...
1,Крёстный отец,1972,Фрэнсис Форд Коппола,"эпический фильм, гангстерский фильм, фильм-тра...",Крёстный оте́ц может означать:
2,Тёмный рыцарь,2008,Кристофер Нолан,"эпический боевик, эпический фильм, супергеройс...",«Тёмный ры́царь» (англ. The Dark Knight) — аме...
3,Крёстный отец 2,1974,Фрэнсис Форд Коппола,"эпический фильм, гангстерский фильм, фильм-тра...",«Крёстный оте́ц 2» (англ. The Godfather Part I...
4,12 разгневанных мужчин,1957,Сидни Люмет,"юридическая драма, психологическая драма, дете...",«12 разгневанных мужчин» (англ. Twelve Angry M...


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Название  250 non-null    object
 1   Год       250 non-null    object
 2   Режиссёр  250 non-null    object
 3   Рейтинг   250 non-null    object
 4   Описание  250 non-null    object
dtypes: object(5)
memory usage: 9.9+ KB
